In [1]:
import ast

# Function to convert code string to AST
def code_to_ast(code_string):
  try:
    return ast.parse(code_string)
  except SyntaxError:
    return None # some code snippets are not valid (python 2 instead of python 3)

In [2]:
### Linearizing the AST (by passing first node) into a list of tokens
def linearize_ast(node, tokens):
  if node is None:
    return

  tokens.append(type(node).__name__)

  # specific node types
  # if it's a name we add the variable name
  if isinstance(node, ast.Name):
    tokens.append(f"VAR_{node.id}")

  # if it's a constant we add a placeholder to not write actual values
  elif isinstance(node, ast.Constant):
    tokens.append("CONST")

  # if it's a function argument we add the argument name
  elif isinstance(node, ast.arg):
    tokens.append(f"ARG_{node.arg}")

  for child in ast.iter_child_nodes(node):
    linearize_ast(child, tokens)

In [3]:
linearized_tree = []
linearize_ast(code_to_ast("def add(a, b): return a + b + 3"), linearized_tree)
print(linearized_tree)

['Module', 'FunctionDef', 'arguments', 'arg', 'ARG_a', 'arg', 'ARG_b', 'Return', 'BinOp', 'BinOp', 'Name', 'VAR_a', 'Load', 'Add', 'Name', 'VAR_b', 'Load', 'Add', 'Constant', 'CONST']


Let's now build a dictionary with Corpora, exactly as we have done before.

In [4]:
from datasets import load_dataset

train_dataset = load_dataset(
  "json",
  data_files = "./../../data/raw/dataset/python/train.jsonl",
  split = "train")
valid_dataset = load_dataset(
  "json",
  data_files = "./../../data/raw/dataset/python/valid.jsonl",
  split = "train")
test_dataset = load_dataset(
  "json",
  data_files = "./../../data/raw/dataset/python/test.jsonl",
  split = "train")

c:\Users\Sean Andreini\Desktop\Unifi\Machine Learning for Software Analysis\code-summarization-mlsa-project\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [5]:
train_linearized_trees = []
for code in train_dataset['code']:
  linearized_tree = []
  linearize_ast(code_to_ast(code), linearized_tree)
  train_linearized_trees.append(linearized_tree)

valid_linearized_trees = []
for code in valid_dataset['code']:
  linearized_tree = []
  linearize_ast(code_to_ast(code), linearized_tree)
  valid_linearized_trees.append(linearized_tree)

test_linearized_trees = []
for code in test_dataset['code']:
  linearized_tree = []
  linearize_ast(code_to_ast(code), linearized_tree)
  test_linearized_trees.append(linearized_tree)

print(train_linearized_trees[0])

['Module', 'FunctionDef', 'arguments', 'arg', 'ARG_p', 'arg', 'ARG_level', 'Constant', 'CONST', 'Expr', 'Constant', 'CONST', 'Assign', 'Name', 'VAR_level', 'Store', 'BinOp', 'Name', 'VAR_level', 'Load', 'Add', 'Constant', 'CONST', 'Assign', 'Name', 'VAR_result', 'Store', 'Call', 'Attribute', 'Name', 'VAR_p', 'Load', 'Load', 'Name', 'VAR_level', 'Load', 'Return', 'BinOp', 'BinOp', 'Subscript', 'Name', 'VAR_result', 'Load', 'Constant', 'CONST', 'Load', 'Add', 'Name', 'VAR_level', 'Load', 'Add', 'Subscript', 'Call', 'Attribute', 'Subscript', 'Name', 'VAR_result', 'Load', 'Constant', 'CONST', 'Load', 'Load', 'Constant', 'CONST', 'Constant', 'CONST', 'Load']


In [6]:
from gensim import corpora
code_dictionary = corpora.Dictionary(train_linearized_trees)
special_tokens = {'[UNK]': 0, '[PAD]': 1, '[BOS]': 2, '[EOS]': 3}
code_dictionary.patch_with_special_tokens(special_tokens)

In [7]:
code_dictionary.token2id

{'ARG_level': 444628,
 'ARG_p': 444629,
 'Add': 444630,
 'Assign': 444631,
 'Attribute': 4,
 'BinOp': 5,
 'CONST': 6,
 'Call': 7,
 'Constant': 8,
 'Expr': 9,
 'FunctionDef': 10,
 'Load': 11,
 'Module': 12,
 'Name': 13,
 'Return': 14,
 'Store': 15,
 'Subscript': 16,
 'VAR_level': 17,
 'VAR_p': 18,
 'VAR_result': 19,
 'arg': 20,
 'arguments': 21,
 'ARG_d': 22,
 'Compare': 23,
 'Eq': 24,
 'ExceptHandler': 25,
 'If': 26,
 'Not': 27,
 'Try': 28,
 'UnaryOp': 29,
 'VAR_OSError': 30,
 'VAR_d': 31,
 'VAR_errno': 32,
 'VAR_msg': 33,
 'VAR_oe': 34,
 'VAR_os': 35,
 'VAR_twdd': 36,
 'ARG_fnh': 37,
 'ARG_mode': 38,
 'Raise': 39,
 'VAR_ValueError': 40,
 'VAR_file': 41,
 'VAR_fnh': 42,
 'VAR_handle': 43,
 'VAR_isinstance': 44,
 'VAR_mode': 45,
 'VAR_open': 46,
 'VAR_str': 47,
 'ARG_categories': 48,
 'ARG_header': 49,
 'ARG_imap': 50,
 'And': 51,
 'Assert': 52,
 'BoolOp': 53,
 'Continue': 54,
 'Dict': 55,
 'For': 56,
 'Gt': 57,
 'In': 58,
 'Is': 59,
 'ListComp': 60,
 'NotIn': 61,
 'Tuple': 62,
 'VAR_As

In [8]:
len(code_dictionary)

444632

We can see that we went from a code_dictionary of over 1 million of tokens (see notebook 02) to just over 400k. This can help speed up the training and give some more accurate results. So, let's try this by just copy-pasting the model from notebook 03. We're gonna use the same vocab for docstrings. We're gonna save the processed dataset and test it on the next notebook.

In [9]:
from datasets import load_from_disk
old_dataset = load_from_disk("../../data/processed/tokenized_codexglue")

In [10]:
train_input_ids = [
  [code_dictionary.token2id.get(token, code_dictionary.token2id['[UNK]']) for token in tree]
  for tree in train_linearized_trees
]

valid_input_ids = [
  [code_dictionary.token2id.get(token, code_dictionary.token2id['[UNK]']) for token in tree]
  for tree in valid_linearized_trees
]

test_input_ids = [
  [code_dictionary.token2id.get(token, code_dictionary.token2id['[UNK]']) for token in tree]
  for tree in test_linearized_trees
]

In [11]:
from datasets import Dataset, DatasetDict

processed_datasets = DatasetDict({
  'train': Dataset.from_dict({
    'input_ids': train_input_ids,
    'labels': old_dataset['train']['labels']
  }),
  'valid': Dataset.from_dict({
    'input_ids': valid_input_ids,
    'labels': old_dataset['valid']['labels']
  }),
  'test': Dataset.from_dict({
    'input_ids': test_input_ids,
    'labels': old_dataset['test']['labels']
  })
})

We're gonna save a copy of our docstring dictionary, so as to have everything organized.

In [12]:
docstring_dictionary = corpora.Dictionary.load('./../../data/processed/tokenized_codexglue/docstring_dictionary.pt')

In [13]:
processed_datasets.save_to_disk('./../../data/processed/ast01/')
code_dictionary.save('./../../data/processed/ast01/code_dictionary.pt')
docstring_dictionary.save('./../../data/processed/ast01/docstring_dictionary.pt')

Saving the dataset (1/1 shards): 100%|██████████| 14918/14918 [00:00<00:00, 596569.80 examples/s]
